# ДЗ 1 — Побить логистическую регрессию на Telco Churn

> Если зачем-то закрыли лекцию: это [Модуль 3](https://itrubnikov.github.io/Train_of_Thought/modules/03-catboost) курса «От нуля до своих агентов».

Цель — за вечер пройти полный пайплайн на табличке: загрузить датасет, обучить логистическую регрессию-бейзлайн, обогнать её CatBoost'ом **без feature engineering'а**, посмотреть SHAP-объяснения.

**Что от вас требуется:** заполнить 3 блока `TODO`. Остальное уже написано. CatBoost и SHAP установятся первой ячейкой.

**Время:** 45—60 минут.

**Как сдавать:** `Файл → Сохранить копию на Диске` → дописать `TODO` → запустить все ячейки → `Поделиться → у кого есть ссылка → Просмотр` → прислать ссылку в чат курса как `[Модуль 3, ДЗ 1] {ссылка}`.

## Шаг 0. Установка библиотек

В Colab нужны только два пакета сверху (`pandas`, `scikit-learn`, `matplotlib` уже стоят).

In [ ]:
!pip install -q catboost shap

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Шаг 1. Данные — Telco Customer Churn

[Датасет на Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn). 7 043 клиента телеком-оператора, 20 фичей (16 категориальных строк + 4 числовых). Таргет — ушёл ли клиент за последний месяц.

Грузим напрямую с GitHub-зеркала IBM-датасета, чтобы не возиться с Kaggle API.

In [ ]:
URL = (
    'https://raw.githubusercontent.com/IBM/'
    'telco-customer-churn-on-icp4d/master/data/'
    'Telco-Customer-Churn.csv'
)
df = pd.read_csv(URL)

# TotalCharges идёт строкой с пробелами вместо NaN — чиним
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.drop(columns=['customerID']).dropna()

print(df.shape)
df.head()

In [ ]:
y = (df.pop('Churn') == 'Yes').astype(int)
X = df

cat_features = X.select_dtypes(include='object').columns.tolist()
num_features = X.select_dtypes(exclude='object').columns.tolist()
print(f'категориальных: {len(cat_features)} → {cat_features}')
print(f'числовых: {len(num_features)} → {num_features}')
print(f'доля положительного класса: {y.mean():.2%}')

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

## Шаг 2. Бейзлайн — логистическая регрессия с one-hot

Готовый код, просто запустите. Это та модель, которую вам нужно обогнать.

Два слова про **что это вообще такое**:

- **Логистическая регрессия** — самая простая обучаемая модель для задачи «да или нет» (ушёл клиент / не ушёл). По сути берёт ваши признаки (возраст, тип контракта, сумму счёта), умножает каждый на свой коэффициент-вес, складывает и пропускает через S-образную кривую, чтобы получить вероятность от 0 до 1. Учится за секунды, объясняется в одну строку, и поэтому это **дефолтный бейзлайн** в табличном ML: если ваша новая навороченная модель не бьёт логрегрессию — она вам не нужна.
- **One-hot encoding** — способ скормить *текстовую* категорию в модель, которая умеет только числа. Колонка `Contract` со значениями `Month-to-month / One year / Two year` превращается в три отдельные числовые колонки, в каждой 0 или 1 («это месячный контракт? — да/нет»). Без этого `LogisticRegression` не знает, что делать со строкой `'Month-to-month'`. У этого подхода есть боли (взрыв размерности на колонках с тысячами уникальных значений, плохая работа с редкими категориями) — про них в [лекции модуля](https://itrubnikov.github.io/Train_of_Thought/modules/03-catboost/). Главная фишка CatBoost'а — что ему всё это руками делать не надо.
- **`StandardScaler`** — приводит числовые колонки к нулевому среднему и единичному разбросу. Линейным моделям это важно, чтобы коэффициенты для «возраста в годах» и «дохода в рублях» жили в одном масштабе. Деревьям (CatBoost и компании) — пофиг, они работают порогами и масштабу безразличны.
- **`ColumnTransformer` + `Pipeline`** — стандартный sklearn-приём, чтобы препроцессинг и сама модель ехали как одно целое: `.fit()` сначала обучает преобразования (one-hot, scaler), потом модель; `.predict()` применяет их в том же порядке. Страховка от типичного бага «забыл прогнать StandardScaler перед predict — получил мусор».
- **AUC** (Area Under ROC Curve) — метрика, которую мы меряем. Грубо: «вероятность того, что модель поставит случайному ушедшему клиенту score выше, чем случайному оставшемуся». 0.5 — наугад, 1.0 — идеально, 0.82 на этом датасете — приличный честный бейзлайн.

In [ ]:
preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
    ('num', StandardScaler(), num_features),
])
logreg = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(max_iter=2000, C=1.0)),
])

t0 = time.time()
logreg.fit(X_tr, y_tr)
logreg_time = time.time() - t0

logreg_auc = roc_auc_score(y_te, logreg.predict_proba(X_te)[:, 1])
print(f'LogReg  AUC = {logreg_auc:.4f}  ({logreg_time:.1f} s)')

## Шаг 3. TODO 1 — CatBoost

Обучите `CatBoostClassifier` так, чтобы:
- получить AUC ≥ 0.84 на тесте,
- передать `cat_features` ЯВНО (иначе будет беда),
- использовать `eval_set=(X_te, y_te)` и `early_stopping_rounds=30`,
- замерить время обучения в `cat_time`.

Подсказка-каркас:

```python
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    cat_features=cat_features,
    eval_metric='AUC',
    early_stopping_rounds=30,
    random_seed=RANDOM_STATE,
    verbose=0,
)
t0 = time.time()
cat_model.fit(X_tr, y_tr, eval_set=(X_te, y_te))
cat_time = time.time() - t0
cat_auc = roc_auc_score(y_te, cat_model.predict_proba(X_te)[:, 1])
```

Перепишите ячейку под себя и **проверьте**, что AUC ≥ 0.84.

In [ ]:
# TODO 1: обучить CatBoostClassifier, сохранить в cat_model, cat_auc, cat_time
raise NotImplementedError('Реализуй TODO 1 — см. инструкцию выше.')

print(f'CatBoost AUC = {cat_auc:.4f}  ({cat_time:.1f} s)')
assert cat_auc >= 0.84, f'AUC {cat_auc:.4f} ниже 0.84 — проверь cat_features и early_stopping_rounds'

In [ ]:
comparison = pd.DataFrame([
    {'model': 'LogReg + OneHot', 'AUC': logreg_auc, 'time, s': round(logreg_time, 2)},
    {'model': 'CatBoost',        'AUC': cat_auc,    'time, s': round(cat_time, 2)},
])
comparison.sort_values('AUC', ascending=False)

## Шаг 4. TODO 2 — SHAP summary plot

Постройте `shap.summary_plot` для CatBoost-модели. Сохраните картинку как `shap_summary.png`.

Что такое SHAP, в одну строку: способ для каждой фичи показать, **насколько и в какую сторону** она в среднем толкает предсказание модели. Глобальная картинка важности + знак влияния.

Каркас:

```python
import shap
explainer = shap.TreeExplainer(cat_model)
shap_values = explainer.shap_values(X_te)
shap.summary_plot(shap_values, X_te, max_display=10, show=False)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=120, bbox_inches='tight')
plt.show()
```

После запуска ответьте одним предложением (в ячейке ниже): **«топ-3 фичи, которые гонят клиента к оттоку, — это …»**.

In [ ]:
# TODO 2: SHAP summary plot для CatBoost-модели
raise NotImplementedError('Реализуй TODO 2 — см. инструкцию выше.')

**Ваш ответ:** топ-3 фичи, которые гонят клиента к оттоку, — это … _(допишите)_

## Шаг 5. TODO 3 — Waterfall plot для одного клиента

`summary_plot` показал общую картину. Теперь зум на одного человека: найдите в `X_te` строку, для которой `cat_model.predict_proba(...)[:, 1] > 0.8` (очень уверенный churn). Постройте для неё waterfall plot, сохраните как `shap_waterfall.png` и опишите одной фразой, что именно в её профиле триггерит модель.

Каркас:

```python
probs = cat_model.predict_proba(X_te)[:, 1]
idx = int(np.argmax(probs))  # самый уверенный churn
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[idx],
        base_values=explainer.expected_value,
        data=X_te.iloc[idx],
        feature_names=X_te.columns.tolist(),
    ),
    show=False,
)
plt.tight_layout()
plt.savefig('shap_waterfall.png', dpi=120, bbox_inches='tight')
plt.show()
print('P(churn) =', probs[idx])
```

In [ ]:
# TODO 3: waterfall plot для одного уверенного churn'а
raise NotImplementedError('Реализуй TODO 3 — см. инструкцию выше.')

**Ваш ответ:** модель уверена в churn'е этого клиента, потому что … _(допишите одной фразой)_

## Что вы только что сделали

За один вечер вы прошли путь, который ещё 10 лет назад занимал у ML-инженера неделю: загрузили реальный бизнес-датасет, обучили две модели (бейзлайн логрегрессии и CatBoost), сравнили их по AUC и времени, объяснили лучшую через SHAP. Это и есть «правая ветка карты ML» из [Модуля 2](https://itrubnikov.github.io/Train_of_Thought/modules/02-ml-map) в действии.

**Чек перед сдачей:**
- [ ] CatBoost AUC ≥ 0.84.
- [ ] В таблице сравнения видны цифры обеих моделей.
- [ ] `shap_summary.png` сохранился и видно осмысленные фичи (`tenure`, `Contract`, `TotalCharges`).
- [ ] `shap_waterfall.png` сохранился и подписан одной фразой.
- [ ] Прислана ссылка в чат курса как `[Модуль 3, ДЗ 1] {ссылка}`.

Дальше — [Модуль 4a: micrograd за час](https://itrubnikov.github.io/Train_of_Thought/modules/04a-micrograd).